# 🎯 Batch Scoring — Energy Grid Outage Prediction

Scores all substations with the trained model to produce outage risk
predictions and regional risk rankings.

**Reads from:** `gold_ml_features`, saved model

**Writes to:** `gold_ml_predictions`, `gold_ml_summary`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count,
    sum as spark_sum, round as spark_round, udf
)
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StringIndexer
from synapse.ml.lightgbm import LightGBMClassificationModel

spark = SparkSession.builder.getOrCreate()
model = LightGBMClassificationModel.load('Files/models/outage_prediction_lgbm')
features_df = spark.read.table('gold_ml_features')
print(f'Scoring {features_df.count():,} feature rows')

In [ ]:
numeric_features = [
    'avg_voltage', 'std_voltage', 'min_voltage', 'max_voltage', 'voltage_range', 'voltage_deviation',
    'avg_frequency', 'std_frequency', 'freq_deviation',
    'avg_power_factor', 'min_power_factor',
    'avg_load', 'max_load',
    'avg_temp', 'max_temp',
    'reading_count', 'day_of_week', 'month',
]

indexer_region = StringIndexer(inputCol='region', outputCol='region_idx', handleInvalid='keep')
indexed_df = indexer_region.fit(features_df).transform(features_df)
all_features = numeric_features + ['region_idx']
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='skip')
model_df = assembler.transform(indexed_df)

In [ ]:
scored = model.transform(model_df)

extract_prob = udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.0, DoubleType())

predictions = (
    scored
    .withColumn('outage_probability', spark_round(extract_prob(col('probability')), 4))
    .withColumn('predicted_outage', col('prediction').cast('int'))
    .withColumn('risk_level',
        when(col('outage_probability') > 0.8, 'critical')
        .when(col('outage_probability') > 0.6, 'high')
        .when(col('outage_probability') > 0.4, 'medium')
        .otherwise('low')
    )
    .withColumn('scored_at', current_timestamp())
    .select(
        'substation_id', 'region', 'sensor_date',
        'avg_voltage', 'avg_frequency', 'avg_power_factor', 'avg_load', 'avg_temp',
        'voltage_deviation', 'freq_deviation',
        'had_outage', 'predicted_outage',
        'outage_probability', 'risk_level',
        'scored_at'
    )
)

predictions.write.mode('overwrite').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count():,} rows')
predictions.groupBy('risk_level').count().orderBy('count', ascending=False).show()

In [ ]:
summary = (
    predictions
    .groupBy('substation_id', 'region')
    .agg(
        spark_round(avg('outage_probability'), 4).alias('avg_outage_risk'),
        spark_sum('predicted_outage').alias('predicted_outage_days'),
        count('*').alias('total_days'),
        spark_round(avg('voltage_deviation'), 2).alias('avg_voltage_deviation'),
        spark_round(avg('freq_deviation'), 4).alias('avg_freq_deviation'),
        spark_round(avg('avg_load'), 2).alias('avg_daily_load'),
    )
    .withColumn('outage_rate', spark_round(col('predicted_outage_days') / col('total_days') * 100, 1))
    .withColumn('overall_risk',
        when(col('avg_outage_risk') > 0.6, 'high')
        .when(col('avg_outage_risk') > 0.3, 'medium')
        .otherwise('low')
    )
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('avg_outage_risk').desc())
)

summary.write.mode('overwrite').format('delta').saveAsTable('gold_ml_summary')
print(f'\nSubstation risk summary: {summary.count()} substations')
summary.show(10, truncate=False)

In [ ]:
spark.sql('OPTIMIZE gold_ml_predictions')
spark.sql('OPTIMIZE gold_ml_summary')
print('\nAll Gold ML tables optimized')

print('\n=== Scoring Summary ===')
print(f'Total predictions: {predictions.count():,}')
print(f'Substations scored: {summary.count()}')
high_risk = summary.filter(col('overall_risk') == 'high').count()
print(f'High-risk substations: {high_risk}')